# 📝 博客发布助手

这个 Notebook 帮你完成三件事：

1. **发布新文章** —— 创建 Markdown 草稿 → 用 VS Code 编辑 → 一键 `git push` 上线
2. **纯文字转 Markdown** —— 把你手写的纯文字稿（`.txt`）自动转成博客用的 Markdown
3. **互动功能原型** —— 在 Notebook 里直接体验「选择题调查」和「聊天区」长什么样，思考怎么落地到博客

> 技术栈：Python（标准库 + ipywidgets），博客是 **Hexo + GitHub Pages**，目录在 `C:\Users\Operator\Desktop\blog`


## 0. 激活环境（重要！）

本 Notebook 用 **博客专属环境** `blog`，运行前请确认：

- 右上角 Kernel 选择 **`Blog (Python 3.13)`**（即 conda 的 `blog` 环境）
- 如果没看到，点击内核选择器 → 选「Blog (Python 3.13)」

```bash
# 在终端确认 blog 环境
conda activate blog
python --version   # 应显示 Python 3.13.x
```

> 为什么用 `blog` 环境？因为它和博客一体：既有 **Python 3.13**（跑本 Notebook 的转换/发布逻辑），又有 **Node.js 26**（跑 Hexo 构建），发布流程全在一个环境里搞定，不污染你的 `STU` 学习环境。
> 本 Notebook 只用标准库 + ipywidgets，不需要额外安装任何包。


In [ ]:
# 1. 导入所需库 + 配置博客路径
# 只用标准库，任何 Python 3 都能跑；ipywidgets 是 Notebook 自带的交互控件

import os            # 处理文件路径
import re            # 正则：用于纯文字转 Markdown 时的规则识别
import datetime      # 生成文章日期
import subprocess    # 执行 git 命令（提交推送）
from IPython.display import display, Markdown   # 在 Notebook 里渲染 Markdown 预览

# 博客根目录（如果博客换位置了，改这一行即可）
BLOG_DIR  = r"C:\Users\Operator\Desktop\blog"
POSTS_DIR = os.path.join(BLOG_DIR, "source", "_posts")   # 文章存放目录

print("✅ 博客路径：", BLOG_DIR)
print("✅ 文章目录：", POSTS_DIR, "存在：", os.path.isdir(POSTS_DIR))

In [ ]:
# 2. 读取纯文字文章
# 把你手写的纯文字稿（.txt / .md 都行）放进来，用这个函数读取

def read_text_file(path: str) -> str:
    """
    读取纯文字文章文件
    - path: 文件路径（可以是绝对路径，或相对博客目录）
    - 返回：文件内容字符串
    """
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

# ── 用法示例 ─────────────────────────────
# 把下面的文件名换成你自己的草稿文件（放博客目录下即可）
DRAFT_PATH = os.path.join(BLOG_DIR, "我的草稿.txt")

if os.path.exists(DRAFT_PATH):
    text = read_text_file(DRAFT_PATH)
    print(f"✅ 已读取 {len(text)} 个字符")
    print("—— 前 200 字预览 ——")
    print(text[:200])
else:
    text = ""
    print("⚠️ 没找到草稿文件，请先创建：", DRAFT_PATH)
    print("   （可以先运行后面的转换函数，用内置示例文字测试）")

In [ ]:
# 3. 纯文字 → Markdown 转换（核心）
# 思路：纯文字和 Markdown 都是"按行写"，所以逐行扫描，用几条规则识别元素：

def text_to_markdown(text: str) -> str:
    """
    把纯文字转成 Markdown（简单规则版）
    识别规则：
      1. 空行          → 保留，作为段落分隔
      2. "第X章/第X节…" → 转成 ## 二级标题
      3. "- * •" 开头   → 转成无序列表
      4. "1. 2. 3." 开头 → 转成有序列表
      5. 其他          → 普通段落，原样保留
    """
    out = []                       # 收集转换后的每一行
    for line in text.splitlines(): # 逐行处理
        s = line.strip()           # 去掉首尾空格方便判断

        if not s:
            out.append("")         # 规则 1：空行 → 段落分隔
        elif re.match(r'^(第[\d一二三四五六七八九十百]+[章节][^。！？]*)$', s) and len(s) <= 25:
            out.append(f"## {s}")  # 规则 2：章/节标题 → 二级标题
        elif re.match(r'^[-*•]\s+', s):
            out.append("- " + re.sub(r'^[-*•]\s+', '', s))     # 规则 3：无序列表
        elif re.match(r'^\d+[.、)]\s*', s):
            out.append(re.sub(r'^(\d+)[.、)]\s*', r'\1. ', s)) # 规则 4：有序列表
        else:
            out.append(s)          # 规则 5：普通段落

    return "\n".join(out)          # 拼回多行文本


# ── 内置示例：没有草稿文件也能先测试 ──
sample_text = """第1章 引言

这是第一段话，介绍博客要写什么内容。

- 要点一
- 要点二

第2章 正文

1. 第一步做什么
2. 第二步做什么

这是结尾的普通段落。"""

md = text_to_markdown(sample_text)
print(md)

In [ ]:
# 4. 展示转换结果（可视化验证）
# 把上一步生成的 Markdown 在 Notebook 里直接渲染出来，检查效果

# 用刚才的示例转一遍（如果有草稿文件，就用草稿内容）
source = text if text else sample_text
md = text_to_markdown(source)

# Markdown() 会像博客一样渲染，直接看排版对不对
display(Markdown(md))

# 同时显示原始代码（方便核对语法）
print("──── 转换后的 Markdown 源码 ────")
print(md)

In [ ]:
# 5. 保存为博客文章 + 一键发布
# 把转换好的 Markdown 存进 source/_posts/，再 git 提交推送 → GitHub Actions 自动部署

def save_post(title: str, content: str) -> str:
    """
    保存为一篇 Hexo 文章（自动加 front matter 模板）
    - title:   文章标题
    - content: Markdown 正文
    - 返回：保存的文件路径
    """
    today = datetime.date.today().isoformat()
    # front matter 是 Hexo 认识文章"元信息"的头部（标题/日期/分类/标签）
    front = (
        "---\n"
        f"title: {title}\n"
        f"date: {today} 12:00:00\n"
        "tags:\n"
        "  - 技术\n"
        "categories:\n"
        "  - 技术\n"
        "---\n\n"
    )
    safe = title.replace("/", "-").replace("\\", "-").replace(":", "：")
    path = os.path.join(POSTS_DIR, f"{safe}.md")
    with open(path, "w", encoding="utf-8") as f:
        f.write(front + content)
    print(f"✅ 文章已保存：{path}")
    return path


def git_publish(title: str):
    """
    提交并推送博客改动（GitHub Actions 会自动构建部署）
    """
    print("🔄 git add / commit / push …")
    # 依次执行三条 git 命令
    cmds = [
        ['git', '-C', BLOG_DIR, 'add', '.'],
        ['git', '-C', BLOG_DIR, 'commit', '-m', f'新文章：{title}'],
        ['git', '-C', BLOG_DIR, 'push'],
    ]
    for cmd in cmds:
        r = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace")
        out = (r.stdout or "").strip()
        err = (r.stderr or "").strip()
        if out: print(out)
        if r.returncode != 0 and "error" in err.lower(): print(err)
    print("🎉 已推送！1-2 分钟后访问 https://gutianshuo.github.io")


# ── 用法：先保存，再发布 ─────────────────
# save_post("我的新文章", md)      # 保存文章（会生成 front matter）
# git_publish("我的新文章")        # 推送上线（也可以再用 VS Code 细化后再推）

# 想用 VS Code 打开刚保存的文章继续编辑：
# os.system('code "' + os.path.join(POSTS_DIR, "我的新文章.md") + '"')

In [ ]:
# 6. 选择题调查原型（ipywidgets）
# 先在这里体验"单选投票"交互，思考怎么落地到博客文章里

import ipywidgets as widgets

# 三个控件：单选按钮 + 提交按钮 + 输出区
question = widgets.RadioButtons(
    options=["很棒 👍", "一般般", "有待改进 👎"],
    description="评价博客：",
    layout=widgets.Layout(width="60%"),
)
submit = widgets.Button(description="提交投票", button_style="primary")
result = widgets.Output()   # 显示反馈的地方

def on_submit(b):
    """点击提交按钮时触发"""
    with result:
        result.clear_output()                      # 清空旧输出
        print(f"✅ 收到！你选了「{question.value}」，谢谢参与～")
        # 真实博客里：这里会把答案发给后端统计（见下面第 8 节的讨论）

submit.on_click(on_submit)                          # 绑定点击事件

# 把控件排列显示出来
display(widgets.VBox([question, submit, result]))

In [ ]:
# 7. 模拟聊天区原型（Widgets）
# 先在这里体验"对话界面"，思考怎么给博客加评论/聊天区

msg_box = widgets.Textarea(
    placeholder="输入你的留言…",
    description="留言：",
    layout=widgets.Layout(width="70%", height="60px"),
)
send_btn = widgets.Button(description="发送", button_style="success")
chat_area = widgets.Output()      # 聊天记录显示区
history = []                      # 保存所有消息

def on_send(b):
    """点击发送：把消息追加到聊天区"""
    text = msg_box.value.strip()
    if not text:
        return                     # 空消息不发送
    history.append(f"你：{text}")
    with chat_area:
        chat_area.clear_output()
        for line in history:       # 逐条显示历史
            print(line)
    msg_box.value = ""             # 清空输入框

send_btn.on_click(on_send)

display(widgets.VBox([msg_box, send_btn, chat_area]))

## 8. 思考：这两个功能怎么落地到博客？

刚才在 Notebook 里体验的交互（投票、聊天）用的是 **Python 内核**，而你的博客是**纯静态网页**（没有服务器），所以不能直接搬过去。以下是可行方案：

### 📊 选择题调查 / 投票

| 方案 | 做法 | 优点 | 缺点 |
|---|---|---|---|
| **Google Forms 嵌入** | 建一个问卷 → 复制嵌入代码贴进文章 | 免费、结果自动汇总成图表 | 国内访问可能不稳 |
| **腾讯问卷** | 同上，国内版 | 国内访问快 | 界面稍重 |
| **Giscus 表情反应** | 评论系统自带的 👍 反应 | 零配置、轻量 | 只能👍👎，不能自定义选项 |
| **自写静态 JS 投票** | 在文章里加一段 `<script>` 单选按钮 | 完全自主、无第三方 | **结果不持久**（刷新即清零，除非接后端） |

> 💡 **推荐**：单题投票用 **Giscus 的 👍/👎 反应**（免费、无后端、数据存 GitHub）；认真做调研用 **腾讯问卷/Google Forms 嵌入**（结果可汇总）。想 100% 自控但接受"纯前端计数"就用自写 JS。

### 💬 聊天区 / 评论区

| 方案 | 原理 | 优点 | 缺点 |
|---|---|---|---|
| **Giscus** ⭐ | 评论存到 **GitHub Discussions** | 免费、无后端、无审查、国内可访问、支持反应和 Markdown | 读者要有 GitHub 账号 |
| **Utterances** | 评论存到 GitHub Issues | 同上，更老牌 | 功能比 Giscus 少 |
| **Valine / Waline** | 需要 LeanCloud / 自托管后端 | 无账号也能评 | 要配后端、有免费额度限制 |
| **Disqus** | 第三方云服务 | 功能全 | 国内访问差、有广告 |

> 💡 **强烈推荐 Giscus**：和你"不想被审查"的需求完美契合——评论数据存你自己的 GitHub 仓库（Discussions），完全自主；只需在仓库设置里启用 Discussions + 安装 Giscus 应用，然后把一段 `<script>` 贴进主题模板即可。**你的 landscape 主题已经内置了 disqus 和 valine，加 Giscus 也很容易。**

### 落地步骤（如果要做 Giscus）
1. 在 `GuTianshuo.github.io` 仓库 → Settings → 启用 **Discussions**
2. 访问 giscus.app → 按引导生成 `<script>` 代码
3. 把代码贴进主题 `layout/_partial/article.ejs`（评论位置）
4. 推送上线，文章底部就会出现评论区 🎉


## 9. 完整使用流程总结

### 日常发布一篇文章
```
① 运行 cell 1-4（导入 + 读取草稿 + 转 Markdown + 预览）
② 检查预览效果，不满意就手动改草稿再转
③ 运行 cell 5 里的 save_post("标题", md)   → 生成博客文章文件
④ （可选）用 VS Code 打开文章继续细化
⑤ 运行 cell 5 里的 git_publish("标题")     → 推送上线
⑥ 1-2 分钟后访问 https://gutianshuo.github.io 刷新查看
```

### 小贴士
- **转换规则是启发式的**：简单标题/列表能自动识别；复杂排版（加粗、链接、表格）建议转换后再用 VS Code 手动补
- **想要更聪明的转换**？把纯文字粘贴给 Copilot 说「帮我转成 Markdown」，AI 理解力更强
- **投票/评论区**：先用 cell 6/7 体验交互逻辑，落地博客用第 8 节的 Giscus + 问卷方案

### 本 Notebook 用到的关键知识点
| 知识点 | 在哪 |
|---|---|
| conda 环境与 kernel | cell 0 |
| 正则表达式（re）做文本识别 | cell 3 |
| Markdown 渲染预览 | cell 4 |
| Hexo front matter 与 git 自动部署 | cell 5 |
| ipywidgets 交互控件 | cell 6 / 7 |
| 静态博客的功能边界（无后端） | cell 8 |
